# PreProcess Stage 2
## Bubble filter

## Prerequisites
Ensure pre_process_stage_1 has been run and the following two files are now available: 


- `intermediate_grouper.json` — the grouped image data
- `pipeline_stats.json` — the timing and summary information

## Select data paths
Define the config.yaml path, the name of the intermediate json and the name of the output stats json. If unsure, do not change the defaults.
If Stage 1 was ran with default settings, these will already be correct

N.B. The **Skip Filter** skips the removal of bubbles 

In [ ]:
!cd ..
CONFIG_PATH = "pre_process/config.yaml"
IN_DATA     = "intermediate_grouper.json"
OUT_DATA    = "intermediate_filtered.json"
STATS_FILE  = "pipeline_stats.json"
SKIP_FILTER = False  # Set to True to skip filtering entirely

## Load Stage 1 Output
The subsequent Cell of code will read your settings, the config.yaml and the grouped images (generated from stage 1)

In [ ]:
import json
from pathlib import Path

from pre_process._pre_process_utils.pipeline_utils import load_config, deserialise_buckets

config = load_config(CONFIG_PATH)
buckets = deserialise_buckets(json.loads(Path(IN_DATA).read_text(encoding="utf-8")))
pipeline_stats = json.loads(Path(STATS_FILE).read_text(encoding="utf-8"))

total = sum(len(v) for v in buckets.values())
print(f"Loaded {total} images across {len(buckets)} buckets")

## Run Bubble filter
Each image is now checked to see whether it should be kept within the dataset or discarded.
If the **Skip Filter** is True, images will not be discarded

In [ ]:
import time
from pre_process.bubble_filter import BubbleFilter, FilterConfig

if SKIP_FILTER:
    print("Filtering skipped — all images passed through")
    filtered_buckets = buckets
    pipeline_stats["timing"]["filter_s"] = None
    pipeline_stats["filter_stats"] = None
else:
    t0 = time.perf_counter()
    bubble_filter = BubbleFilter(FilterConfig.from_dict(config))
    filtered_buckets, _ = bubble_filter.filter_buckets(buckets)
    dt = time.perf_counter() - t0

    stats = bubble_filter.stats
    print(f"Filter complete in {dt:.1f}s")
    print(f"  Passed:   {stats['passed']}")
    print(f"  Rejected: {stats['rejected']}")
    print(f"  Errors:   {stats['errors']}")

    pipeline_stats["timing"]["filter_s"] = round(dt, 2)
    pipeline_stats["filter_stats"] = stats

## Export Results
Writes filtered dataset and the updated stats to the local disk

In [ ]:
from pre_process._pre_process_utils.pipeline_utils import safe_write_json

if not filtered_buckets:
    print("No images remain after filtering. Nothing to save.")
else:
    serialisable = {str(k): v for k, v in filtered_buckets.items()}
    safe_write_json(serialisable, OUT_DATA)
    safe_write_json(pipeline_stats, STATS_FILE)
    print(f"Filtered data saved to: {OUT_DATA}")
    print(f"Stats updated in:       {STATS_FILE}")

## (Opt) get discard count
Check how many images were removed from the original dataset

In [ ]:
print(f"{'Bucket':<20} {'Before':>8} {'After':>8} {'Rejected':>10}")
print("-" * 50)
for key in buckets:
    before = len(buckets[key])
    after  = len(filtered_buckets.get(key, []))
    print(f"{str(key):<20} {before:>8} {after:>8} {before - after:>10}")


## Troubleshooting

### Cannot find the input files

Stage 2 needs the files that Stage 1 created. If you saved Stage 1's output to a different location or with different names, update the paths in Cell 1 to match.

### Running out of GPU memory

The filter uses an AI model that runs on your graphics card if one is available. If your card does not have enough memory, you can switch to CPU mode by editing your settings file:

```
config.yaml
```

Look for the bubble filter section and change the device setting to

```
cpu
```

You can also try reducing the batch size if one is listed there. See the [PyTorch CUDA semantics documentation](https://docs.pytorch.org/docs/stable/notes/cuda.html) for more details on device management.

### Every image was rejected

If nothing passes the filter, the sensitivity may be set too high for your data. In your settings file, look for a threshold or cutoff value in the bubble filter section and try increasing it (a higher threshold means the filter is more lenient).

---

---

## Developer Notes

This section is intended for developers and technically curious users who want to understand what the data is doing at this stage and why.

### What is tying the data together?

- The **resolution bucket key** still organises the data, inherited from Stage 1. But the concept that matters at this stage is different: the autoencoder treats each image as an **independent sample** to be scored. The bucket structure is a batching convenience, not a semantic requirement — the filter would produce identical results if all images were in a single flat list.

### Is the layout defined by a single point of view?

- The filter views every image through a single lens: "is this a bubble/artefact or a plankton?" This is a binary classification applied uniformly regardless of resolution, provenance, or timestamp. The underlying technique — [reconstruction-error-based anomaly detection](https://www.tensorflow.org/tutorials/generative/autoencoder) — trains on normal samples and flags inputs that the model cannot reconstruct well (see also [Sakurada & Yairi, 2014](https://dl.acm.org/doi/10.1145/2689746.2689747)).
- The data could be recut for **per-archive rejection rates** (to flag a dirty lens or malfunctioning instrument), **score distributions per bucket** (to detect resolution-dependent bias in the autoencoder), or **temporal rejection patterns** (to identify time windows with poor sample quality). These views require only the score and provenance fields that already exist in the records.

### What makes this data uniquely important?

- This is the **decision boundary** of the pipeline. Every image that passes this stage will be written to [OME-Zarr](https://ngff.openmicroscopy.org/latest/) — every image rejected here is gone permanently. The second return value from
  ```python
  bubble_filter.filter_buckets(buckets)
  ```
  (the rejected set, assigned to `_` in the script) is the only opportunity to inspect what was discarded and why.

### Memory layout and access patterns

- **Smallest addressable unit:** a single image, deserialised back into an
  ```
  ndarray
  ```
  from the nested-list JSON representation. The
  ```python
  deserialise_buckets()
  ```
  call is the cost of the Stage 1 → Stage 2 serialisation boundary. [`Path.read_text()`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text) + `json.loads()` is the idiomatic Python pattern for this; it automatically manages the file handle.
- **Read usage:** the autoencoder consumes **every pixel** of every image — unlike Stage 1's grouper, which only read
  ```
  .shape
  ```
  This is full-tensor inference. The entire pixel payload is loaded into GPU (or CPU) memory as a batch.
- **Access frequency:** each image is scored exactly **once**. There is no iterative refinement or multi-pass scoring.
- **Access pattern:** **sequential within each bucket**, but the batch size determines how many images are in flight at once. Within a batch the convolution kernels stride across the full spatial extent of each tensor. The theoretical access pattern is irregular, but in practice GPU libraries such as cuDNN use tiling, shared memory, and data-layout strategies (e.g. NHWC) to achieve [structured, coalesced memory access](https://docs.nvidia.com/deeplearning/performance/dl-performance-convolutional/index.html).
- **Modification:** images are **never modified**. The filter only partitions them into two sets (passed and rejected). The pixel data is carried through byte-for-byte unchanged.

### Timing

- Inference wall-clock time is measured with [`time.perf_counter()`](https://docs.python.org/3/library/time.html#time.perf_counter), which provides the highest-resolution monotonic clock available and is the recommended choice for benchmarking durations in Python 3.12+.

### Who does this data matter to?

- The immediate consumer is **Stage 3** (Zarr writer), which expects clean, artefact-free images. Bubbles and debris that slip through here become persistent entries in the [Zarr](https://zarr.readthedocs.io/) arrays.
- The **rejected set** matters to anyone validating filter quality — instrument technicians checking for hardware issues, or researchers tuning the autoencoder threshold. The current script discards it, but in the notebook you can capture it for inspection by replacing
  ```python
  filtered_buckets, _ = bubble_filter.filter_buckets(buckets)
  ```
  with
  ```python
  filtered_buckets, rejected_buckets = bubble_filter.filter_buckets(buckets)
  ```
- **Latency constraint:** autoencoder inference time dominates this stage. The practical bottleneck is GPU throughput (or CPU fallback). Batch size is the primary tuning lever — larger batches amortise kernel launch overhead but require more VRAM. See the [NVIDIA deep learning performance guide](https://docs.nvidia.com/deeplearning/performance/dl-performance-convolutional/index.html) for details on batch-size and data-layout trade-offs.

### What is implicit?

- The **autoencoder model weights** encode a learned definition of "bubble" that is not expressed anywhere in the config or the data. The threshold in
  ```
  FilterConfig
  ```
  controls sensitivity, but the feature space is entirely implicit in the trained model.
- The **second return value** from
  ```python
  filter_buckets()
  ```
  carries the reconstruction-error scores that motivated each rejection — this information is computed but not persisted by default.
- The **order of images within the output buckets** preserves input order, meaning chronological ordering (if it survived from Stage 1) is maintained. This is relied upon implicitly but not enforced contractually.
- **Model loading security:** if the filter loads weights via [`torch.load`](https://docs.pytorch.org/docs/stable/generated/torch.load.html), note that since PyTorch 2.6 the default is `weights_only=True`, which restricts deserialisation to tensors and primitive types. Custom classes in the checkpoint must be explicitly allow-listed via `torch.serialization.add_safe_globals()`. Ensure you are running **PyTorch ≥ 2.6.0** to benefit from this protection (earlier versions are affected by [CVE-2025-32434](https://docs.pytorch.org/docs/stable/notes/serialization.html)).
